In [ ]:
import socket
import threading

# Dictionary to store client connections and their usernames
clients = {}
usernames = {}

# Broadcast message to all clients
def broadcast(message, client_socket):
    for client in clients:
        if client != client_socket:
            try:
                client.send(message.encode('utf-8'))
            except:
                continue

# Handle client communication
def handle_client(client_socket):
    # Authentication step
    client_socket.send("Enter username: ".encode('utf-8'))
    username = client_socket.recv(1024).decode('utf-8')
    
    # Ensure the username is unique
    while username in usernames:
        client_socket.send("Username already taken, please choose another: ".encode('utf-8'))
        username = client_socket.recv(1024).decode('utf-8')
    
    usernames[client_socket] = username
    clients[client_socket] = username
    
    print(f"{username} has connected.")
    
    # Welcome message and broadcast that a new client has joined
    client_socket.send(f"Welcome {username}!\n".encode('utf-8'))
    broadcast(f"{username} has joined the chat!", client_socket)
    
    while True:
        try:
            message = client_socket.recv(1024).decode('utf-8')
            
            # Check for server commands
            if message.startswith('/'):
                handle_command(message, client_socket)
            else:
                broadcast(f"{username}: {message}", client_socket)
        
        except:
            # Remove client from dictionary if connection is lost
            del clients[client_socket]
            del usernames[client_socket]
            client_socket.close()
            broadcast(f"{username} has left the chat.", client_socket)
            break

# Handle server commands like private messages and disconnecting clients
def handle_command(message, client_socket):
    command = message.split(' ', 1)
    
    if command[0] == '/msg' and len(command) > 1:
        # Private messaging
        recipient_username, msg = command[1].split(' ', 1)
        recipient_socket = None
        
        # Find recipient socket
        for socket, username in usernames.items():
            if username == recipient_username:
                recipient_socket = socket
                break
        
        if recipient_socket:
            recipient_socket.send(f"Private message from {usernames[client_socket]}: {msg}".encode('utf-8'))
            client_socket.send(f"Private message sent to {recipient_username}: {msg}".encode('utf-8'))
        else:
            client_socket.send(f"User {recipient_username} not found.".encode('utf-8'))
    
    elif command[0] == '/disconnect':
        # Disconnect the client
        username = usernames[client_socket]
        client_socket.send(f"Disconnecting {username}...".encode('utf-8'))
        del clients[client_socket]
        del usernames[client_socket]
        client_socket.close()
        broadcast(f"{username} has been disconnected by the server.", client_socket)

# Set up the server to listen for incoming client connections
def start_server():
    server = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    server.bind(('127.0.0.1', 5556))
    server.listen(5)
    print("Server is listening for connections...")
    
    while True:
        client_socket, addr = server.accept()
        print(f"Connection from {addr} established.")
        
        # Start a new thread for each client
        thread = threading.Thread(target=handle_client, args=(client_socket,))
        thread.start()

# Run the server in the current cell (Blocking)
start_server()


Server is listening for connections...
Connection from ('127.0.0.1', 62829) established.
Riri has connected.
